# 🧊 PySpark + Apache Iceberg + MinIO
Exemplo completo: criar namespace, tabela Iceberg, inserir dados e consultar.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
import datetime

spark = (
    SparkSession.builder
    .appName("iceberg-demo")
    # Usa as configs do spark-defaults.conf, mas podemos sobrescrever aqui:
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "hadoop")   # sem REST catalog por ora
    .config("spark.sql.catalog.lakehouse.warehouse", "s3a://lakehouse/warehouse")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin_minio")
    .config("spark.hadoop.fs.s3a.secret.key", "Grunt9-Relenting6-Hula5-Catcall9-Residue3")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ✅")

## 1️⃣ Criar namespace (banco) e tabela Iceberg

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.bronze")

spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.bronze.vendas (
        id          INT,
        produto     STRING,
        quantidade  INT,
        valor       DOUBLE,
        data_venda  DATE
    )
    USING iceberg
    PARTITIONED BY (data_venda)
    TBLPROPERTIES (
        'write.format.default' = 'parquet',
        'write.target-file-size-bytes' = '134217728'
    )
""")

print("Tabela criada ✅")

## 2️⃣ Inserir dados

In [ ]:
from pyspark.sql import Row

dados = [
    Row(id=1, produto="Notebook",  quantidade=2,  valor=4500.0, data_venda=datetime.date(2024, 1, 10)),
    Row(id=2, produto="Mouse",     quantidade=10, valor=150.0,  data_venda=datetime.date(2024, 1, 10)),
    Row(id=3, produto="Teclado",   quantidade=5,  valor=300.0,  data_venda=datetime.date(2024, 1, 15)),
    Row(id=4, produto="Monitor",   quantidade=3,  valor=2100.0, data_venda=datetime.date(2024, 2, 1)),
]

df = spark.createDataFrame(dados)
df.writeTo("lakehouse.bronze.vendas").append()
print("Dados inseridos ✅")

## 3️⃣ Consultar + Time Travel

In [ ]:
# Consulta normal
spark.sql("SELECT * FROM lakehouse.bronze.vendas ORDER BY data_venda").show()

# Histórico de snapshots (time travel)
spark.sql("SELECT snapshot_id, committed_at, operation FROM lakehouse.bronze.vendas.snapshots").show(truncate=False)

# Estatísticas dos arquivos
spark.sql("SELECT file_path, record_count, file_size_in_bytes FROM lakehouse.bronze.vendas.files").show(truncate=False)

## 4️⃣ Upsert com MERGE INTO (Iceberg nativo)

In [ ]:
# Cria tabela de staging
novos = spark.createDataFrame([
    Row(id=2, produto="Mouse Pro", quantidade=15, valor=200.0, data_venda=datetime.date(2024, 1, 10)),  # update
    Row(id=5, produto="Webcam",    quantidade=7,  valor=450.0, data_venda=datetime.date(2024, 3, 1)),   # insert
])
novos.createOrReplaceTempView("staging")

spark.sql("""
    MERGE INTO lakehouse.bronze.vendas AS t
    USING staging AS s
        ON t.id = s.id
    WHEN MATCHED THEN
        UPDATE SET *
    WHEN NOT MATCHED THEN
        INSERT *
""")

spark.sql("SELECT * FROM lakehouse.bronze.vendas ORDER BY id").show()
print("MERGE concluído ✅")

## 5️⃣ Compactação + limpeza de snapshots antigos

In [ ]:
# Compactar pequenos arquivos
spark.sql("""
    CALL lakehouse.system.rewrite_data_files(
        table => 'bronze.vendas',
        strategy => 'binpack',
        options => map('target-file-size-bytes','134217728')
    )
""").show()

# Expirar snapshots com mais de 1 dia (em produção use 7+ dias)
from datetime import datetime, timedelta
cutoff = int((datetime.utcnow() - timedelta(days=1)).timestamp() * 1000)

spark.sql(f"""
    CALL lakehouse.system.expire_snapshots(
        table => 'bronze.vendas',
        older_than => TIMESTAMP '{datetime.utcnow().isoformat()}',
        retain_last => 2
    )
""").show()